# SRPO Reward Diagnostic Study

This notebook diagnoses whether the SRPO world-progress reward mechanism
produces meaningful, progress-aware reward signals.

**Experiments:**

1. **Distance & Reward Separation** — Do demos, SFT successes, SFT failures,
   and random trajectories have distinguishable distances/rewards?
2. **StandardScaler Impact** — Does applying StandardScaler before DBSCAN
   (matching siiRL production code) improve cluster quality?
3. **Reward Formula Comparison** — siiRL (min-max + 0.6 cap) vs. our
   z-score + alpha formulation.
4. **Progress Monotonicity** — Do trajectories with more task progress
   receive monotonically higher rewards?
5. **Per-Frame vs Clip Encoding** — Does V-JEPA 2's 64-frame video-clip
   encoding provide better separation than per-frame mean-pooling?
5b. **Chunked (Sliding-Window) Encoding** — Does splitting trajectories
   into 32-frame windows and encoding each as a clip yield a dense,
   monotonic progress signal?
6. **Embedding Dimensionality** — How many effective dimensions does the
   embedding space have? Is variance concentrated in a few dims?
7. **k-NN Elbow** — Is there natural cluster structure in the embedding space?
8. **Progress Correlation Comparison** — How does SRPO's progress
   correlation compare against Top-Rewarder's published ~0.95?

In [ ]:
import logging
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from vla.diagnostics.collect_trajectories import (
    CollectionConfig,
    collect_demo_trajectories,
    collect_progress_trajectories,
    collect_rollouts,
)
from vla.diagnostics.clustering import (
    ClusteringConfig,
    SOURCE_COLORS,
    SOURCE_DEMO,
    SOURCE_FAILED,
    SOURCE_RANDOM,
    SOURCE_SFT_SUCCESS,
    get_or_compute_embeddings,
    fit_umap,
    plot_panel_b,
)
from vla.diagnostics.reward_analysis import (
    build_distance_table,
    build_progress_correlation_table,
    build_progress_table,
    chunk_trajectory_images,
    cluster_reference_chunks,
    compare_encoding_methods,
    compute_chunked_intra_trajectory_correlation,
    compute_chunked_progress_curve,
    compute_cluster_centers_raw,
    compute_cluster_centers_siirl,
    compute_per_frame_distances,
    compute_progress_correlation,
    compute_siirl_rewards,
    compute_zscore_rewards,
    distances_to_nearest_center,
    embedding_dimension_stats,
    encode_trajectories_clip,
    encode_trajectories_per_frame,
    pca_explained_variance,
    plot_chunked_progress_curves,
    plot_cosine_similarity_matrix,
    plot_dimension_variance,
    plot_distance_kde,
    plot_knn_elbow,
    plot_knn_elbow_comparison,
    plot_pca_explained_variance,
    plot_per_frame_evolution,
    plot_progress_curve,
    plot_reward_kde,
)
from vla.models.world_model import build_world_model
from vla.constants import WorldModelType
from vla.utils import get_device, seed_everything

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
warnings.filterwarnings("ignore", category=FutureWarning)

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## Configuration

In [ ]:
SEED = 42
seed_everything(SEED)
device = get_device()

SUITE = "spatial"
TASK_ID = 2
CACHE_DIR = Path("cache/reward_study")

collect_cfg = CollectionConfig(
    libero_suite=SUITE,
    task_id=TASK_ID,
    num_demos=50,
    num_rollouts=50,
    num_envs=4,
    max_steps=300,
    seed=SEED,
    cache_dir=CACHE_DIR,
)

cluster_cfg = ClusteringConfig(
    subsample_every=1,
    dbscan_min_samples=2,
    dbscan_auto_eps=True,
    dbscan_percentile=25,
    seed=SEED,
    cache_dir=CACHE_DIR,
)

print(f"Device: {device}")
print(f"Task:   {SUITE} task {TASK_ID}")
print(f"Cache:  {CACHE_DIR.resolve()}")

## Step 1 — Collect Trajectories

All buffers are cached after first run.

In [ ]:
demos = collect_demo_trajectories(collect_cfg)
sft_success, sft_failed, random_trajs = collect_rollouts(collect_cfg, device)

print(f"Demos:       {len(demos)}")
print(f"SFT success: {len(sft_success)}")
print(f"SFT failed:  {len(sft_failed)}")
print(f"Random:      {len(random_trajs)}")

## Step 2 — Compute Embeddings (V-JEPA 2)

In [ ]:
encoder = build_world_model(WorldModelType.VJEPA2, device=str(device))

emb_demo = get_or_compute_embeddings(
    demos, encoder, CACHE_DIR / "emb_demos.pt", cluster_cfg.subsample_every,
)
emb_sft_ok = get_or_compute_embeddings(
    sft_success, encoder, CACHE_DIR / "emb_sft_success.pt", cluster_cfg.subsample_every,
)
emb_sft_fail = get_or_compute_embeddings(
    sft_failed, encoder, CACHE_DIR / "emb_sft_failed.pt", cluster_cfg.subsample_every,
)
emb_random = get_or_compute_embeddings(
    random_trajs, encoder, CACHE_DIR / "emb_random.pt", cluster_cfg.subsample_every,
)

for name, emb in [("Demo", emb_demo), ("SFT ok", emb_sft_ok), ("SFT fail", emb_sft_fail), ("Random", emb_random)]:
    print(f"{name:>10s}: {emb.shape}")

In [ ]:
X_demo = emb_demo.cpu().numpy()
X_sft_ok = emb_sft_ok.cpu().numpy()
X_sft_fail = emb_sft_fail.cpu().numpy()
X_random = emb_random.cpu().numpy()

X_all = np.concatenate([X_demo, X_sft_ok, X_sft_fail, X_random], axis=0)
sources = (
    [SOURCE_DEMO] * len(X_demo)
    + [SOURCE_SFT_SUCCESS] * len(X_sft_ok)
    + [SOURCE_FAILED] * len(X_sft_fail)
    + [SOURCE_RANDOM] * len(X_random)
)
print(f"Total embeddings: {X_all.shape}")

---

## Experiment 1 — Distance & Reward Separation

**Hypothesis:** Successful trajectories (demos, SFT success) should cluster
tightly and have small distances to cluster centers. Failed and random
trajectories should be further away. If there's no separation, the SRPO
reward signal is not informative.

In [ ]:
X_reference = np.concatenate([X_demo, X_sft_ok], axis=0)
centers_raw = compute_cluster_centers_raw(X_reference)

dist_groups_raw = {
    "Demo": distances_to_nearest_center(X_demo, centers_raw),
    "SFT Success": distances_to_nearest_center(X_sft_ok, centers_raw),
    "SFT Failed": distances_to_nearest_center(X_sft_fail, centers_raw),
    "Random": distances_to_nearest_center(X_random, centers_raw),
}

print(f"Cluster centers (raw DBSCAN): {centers_raw.shape[0]}")
print()

table_raw = build_distance_table(dist_groups_raw, reward_fn=compute_siirl_rewards)
display(table_raw.style.format(precision=4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

plot_distance_kde(dist_groups_raw, title="Exp 1 — Distance to Nearest Center (raw DBSCAN)", ax=axes[0])
plot_reward_kde(
    dist_groups_raw,
    reward_fn=compute_siirl_rewards,
    title="Exp 1 — siiRL Shaped Rewards (raw DBSCAN)",
    ax=axes[1],
)
plt.tight_layout()
plt.show()

### UMAP Visualisation

In [ ]:
xy = fit_umap(X_all, cluster_cfg)

fig, ax = plt.subplots(figsize=(10, 8))
plot_panel_b(ax, xy, sources)
ax.set_title("UMAP by Trajectory Source", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### Cosine Similarity Between Groups

Ideally: high intra-group similarity for demos/SFT success, lower similarity
with failed/random.

In [ ]:
emb_groups = {
    "Demo": X_demo,
    "SFT Success": X_sft_ok,
    "SFT Failed": X_sft_fail,
    "Random": X_random,
}

fig, ax = plt.subplots(figsize=(7, 6))
plot_cosine_similarity_matrix(emb_groups, ax=ax)
plt.tight_layout()
plt.show()

---

## Experiment 2 — StandardScaler Impact

**Hypothesis:** siiRL applies `StandardScaler` before DBSCAN, which
normalises V-JEPA 2 embedding dimensions and improves cluster quality.
Without it, high-variance dimensions dominate, making all trajectories
look similar.

In [ ]:
centers_siirl, scaler = compute_cluster_centers_siirl(X_reference)

dist_groups_siirl = {
    "Demo": distances_to_nearest_center(X_demo, centers_siirl, scaler),
    "SFT Success": distances_to_nearest_center(X_sft_ok, centers_siirl, scaler),
    "SFT Failed": distances_to_nearest_center(X_sft_fail, centers_siirl, scaler),
    "Random": distances_to_nearest_center(X_random, centers_siirl, scaler),
}

print(f"Cluster centers (siiRL method): {centers_siirl.shape[0]}")
print()

table_siirl = build_distance_table(dist_groups_siirl, reward_fn=compute_siirl_rewards)
display(table_siirl.style.format(precision=4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

plot_distance_kde(
    dist_groups_siirl,
    title="Exp 2 — Distance (StandardScaler + DBSCAN)",
    ax=axes[0],
)
plot_reward_kde(
    dist_groups_siirl,
    reward_fn=compute_siirl_rewards,
    title="Exp 2 — siiRL Rewards (StandardScaler + DBSCAN)",
    ax=axes[1],
)
plt.tight_layout()
plt.show()

In [ ]:
comparison = pd.concat(
    [
        build_distance_table(dist_groups_raw, compute_siirl_rewards).assign(Method="Raw DBSCAN"),
        build_distance_table(dist_groups_siirl, compute_siirl_rewards).assign(Method="siiRL (StandardScaler)"),
    ],
    ignore_index=True,
)
display(
    comparison[["Method", "Group", "N", "Dist Mean", "Dist Std", "Reward Mean", "Reward Std"]]
    .style.format(precision=4)
)

---

## Experiment 3 — Reward Formula Comparison

**Hypothesis:** The siiRL formula (`0.6 * sigmoid(10 * (0.5 - normdist))`)
with min-max normalisation may provide better separation than our z-score
formula (`alpha * sigmoid(-z)`).

In [ ]:
all_dists_siirl = np.concatenate([v for v in dist_groups_siirl.values()])

siirl_rewards = compute_siirl_rewards(all_dists_siirl)
zscore_rewards = compute_zscore_rewards(all_dists_siirl, alpha=0.8)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(all_dists_siirl, siirl_rewards, alpha=0.3, s=10, label="siiRL")
axes[0].set_xlabel("Distance")
axes[0].set_ylabel("Reward")
axes[0].set_title("siiRL: 0.6 * sigmoid(10*(0.5 - norm_dist))")
axes[0].grid(True, alpha=0.3)

axes[1].scatter(all_dists_siirl, zscore_rewards, alpha=0.3, s=10, color="orange", label="z-score")
axes[1].set_xlabel("Distance")
axes[1].set_ylabel("Reward")
axes[1].set_title("Ours: 0.8 * sigmoid(-z_score)")
axes[1].grid(True, alpha=0.3)

plt.suptitle("Exp 3 — Reward Formula Comparison (on siiRL distances)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
formula_rows = []
for name, dists in dist_groups_siirl.items():
    r_siirl = compute_siirl_rewards(dists)
    r_zscore = compute_zscore_rewards(dists, alpha=0.8)
    formula_rows.append({
        "Group": name,
        "N": len(dists),
        "siiRL Mean": r_siirl.mean(),
        "siiRL Std": r_siirl.std(),
        "z-score Mean": r_zscore.mean(),
        "z-score Std": r_zscore.std(),
        "siiRL Range": r_siirl.max() - r_siirl.min(),
        "z-score Range": r_zscore.max() - r_zscore.min(),
    })

formula_df = pd.DataFrame(formula_rows)
display(formula_df.style.format(precision=4))

---

## Experiment 4 — Progress Monotonicity

**Hypothesis:** Trajectories that complete more of the task (higher progress %)
should receive higher rewards. If the V-JEPA 2 embedding captures task
progress, we expect a monotonically decreasing distance (and increasing
reward) as progress increases.

**Method:** For each demo trajectory, we replay the first N% of its recorded
actions (resetting the env first), then switch to random actions for the rest.
Progress levels: 0%, 25%, 50%, 75%, 100%.

In [ ]:
PROGRESS_LEVELS = [1.0, 0.75, 0.5, 0.25, 0.0]

progress_trajs = collect_progress_trajectories(
    cfg=collect_cfg,
    reference_trajs=demos[:20],
    progress_levels=PROGRESS_LEVELS,
    source_name="demos",
)

for level in PROGRESS_LEVELS:
    trajs = progress_trajs[level]
    mean_len = np.mean([t.length for t in trajs]) if trajs else 0
    print(f"Progress {level*100:3.0f}%: {len(trajs)} trajectories, mean length {mean_len:.0f}")

In [ ]:
progress_embs = {}
for level in PROGRESS_LEVELS:
    pct = int(level * 100)
    cache_name = CACHE_DIR / f"emb_progress_{pct}.pt"
    progress_embs[level] = get_or_compute_embeddings(
        progress_trajs[level], encoder, cache_name, cluster_cfg.subsample_every,
    )
    print(f"Progress {pct:3d}%: {progress_embs[level].shape}")

In [ ]:
distances_per_level_raw = {}
distances_per_level_siirl = {}

for level in PROGRESS_LEVELS:
    X_prog = progress_embs[level].cpu().numpy()
    distances_per_level_raw[level] = distances_to_nearest_center(X_prog, centers_raw)
    distances_per_level_siirl[level] = distances_to_nearest_center(X_prog, centers_siirl, scaler)

print("Progress table (siiRL distances):")
progress_df = build_progress_table(PROGRESS_LEVELS, distances_per_level_siirl)
display(progress_df.style.format(precision=4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plot_progress_curve(
    PROGRESS_LEVELS,
    distances_per_level_raw,
    title="Exp 4 — Distance vs Progress (raw DBSCAN)",
    ax=axes[0],
)
plot_progress_curve(
    PROGRESS_LEVELS,
    distances_per_level_siirl,
    title="Exp 4 — Distance vs Progress (siiRL method)",
    ax=axes[1],
)

plt.suptitle("Progress Monotonicity Test", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
corr_raw, pval_raw = compute_progress_correlation(PROGRESS_LEVELS, distances_per_level_raw)
corr_siirl, pval_siirl = compute_progress_correlation(PROGRESS_LEVELS, distances_per_level_siirl)

print(f"Spearman correlation (raw):   r={corr_raw:.4f}, p={pval_raw:.4f}")
print(f"Spearman correlation (siiRL): r={corr_siirl:.4f}, p={pval_siirl:.4f}")
print()
print("Expected: negative correlation (higher progress → lower distance)")

---

## Experiment 5 — Per-Frame vs Clip Encoding

**Hypothesis:** V-JEPA 2 supports two encoding modes: (a) encode each
frame independently and mean-pool, or (b) pass all frames as a single
64-frame video clip with temporal attention. The clip mode captures
temporal context (e.g., object movement trajectory) that per-frame
pooling misses, potentially yielding better separation.

In [ ]:
from vla.utils.tensor import to_float01

all_traj_images = (
    [to_float01(t.images[:t.length]) for t in demos]
    + [to_float01(t.images[:t.length]) for t in sft_success]
    + [to_float01(t.images[:t.length]) for t in sft_failed]
    + [to_float01(t.images[:t.length]) for t in random_trajs]
)

group_labels = (
    ["Demo"] * len(demos)
    + ["SFT Success"] * len(sft_success)
    + ["SFT Failed"] * len(sft_failed)
    + ["Random"] * len(random_trajs)
)

pf_cache = CACHE_DIR / "emb_per_frame_all.pt"
clip_cache = CACHE_DIR / "emb_clip_all.pt"

if pf_cache.exists():
    emb_per_frame_all = torch.load(pf_cache, weights_only=False)
else:
    emb_per_frame_all = torch.from_numpy(
        encode_trajectories_per_frame(all_traj_images, encoder, cluster_cfg.subsample_every)
    )
    torch.save(emb_per_frame_all, pf_cache)

if clip_cache.exists():
    emb_clip_all = torch.load(clip_cache, weights_only=False)
else:
    emb_clip_all = torch.from_numpy(
        encode_trajectories_clip(all_traj_images, encoder, cluster_cfg.subsample_every)
    )
    torch.save(emb_clip_all, clip_cache)

X_pf = emb_per_frame_all.cpu().numpy() if isinstance(emb_per_frame_all, torch.Tensor) else emb_per_frame_all
X_clip = emb_clip_all.cpu().numpy() if isinstance(emb_clip_all, torch.Tensor) else emb_clip_all

print(f"Per-frame embeddings: {X_pf.shape}")
print(f"Clip embeddings:      {X_clip.shape}")

In [ ]:
n_demo = len(demos)
n_ok = len(sft_success)
ref_mask = np.array([True] * (n_demo + n_ok) + [False] * (len(sft_failed) + len(random_trajs)))

centers_pf_raw = compute_cluster_centers_raw(X_pf[ref_mask])
centers_clip_raw = compute_cluster_centers_raw(X_clip[ref_mask])
centers_pf_siirl, scaler_pf = compute_cluster_centers_siirl(X_pf[ref_mask])
centers_clip_siirl, scaler_clip = compute_cluster_centers_siirl(X_clip[ref_mask])

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax_row, (method, c_pf, c_clip, sc_pf, sc_clip) in enumerate([
    ("Raw DBSCAN", centers_pf_raw, centers_clip_raw, None, None),
    ("siiRL (StandardScaler)", centers_pf_siirl, centers_clip_siirl, scaler_pf, scaler_clip),
]):
    for col, (enc_name, X_enc, centers, sc) in enumerate([
        ("Per-Frame", X_pf, c_pf, sc_pf),
        ("64-Frame Clip", X_clip, c_clip, sc_clip),
    ]):
        dist_groups = {}
        offset = 0
        for grp, n in [("Demo", n_demo), ("SFT Success", n_ok), ("SFT Failed", len(sft_failed)), ("Random", len(random_trajs))]:
            dist_groups[grp] = distances_to_nearest_center(X_enc[offset:offset+n], centers, sc)
            offset += n
        plot_distance_kde(dist_groups, title=f"{enc_name} / {method}", ax=axes[ax_row, col])

plt.suptitle("Exp 5 — Per-Frame vs Clip Encoding", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
enc_df = compare_encoding_methods(
    per_frame_embs=X_pf,
    clip_embs=X_clip,
    reference_per_frame=X_pf[ref_mask],
    reference_clip=X_clip[ref_mask],
    group_labels=group_labels,
)
display(enc_df.style.format(precision=4))

---

### Experiment 5b — Chunked (Sliding-Window) Encoding

**Hypothesis:** Instead of encoding a full trajectory into one embedding
(which averages out temporal progress) or individual frames (which lack
temporal context), we split each trajectory into overlapping windows of
~32 frames and encode each window as a V-JEPA 2 video clip.

This yields a **dense, per-chunk distance signal**: early chunks should
be far from success cluster centers, late chunks should be close — giving
a progress-based reward that can be computed at any point during execution.

In [ ]:
WINDOW_SIZE = 32
STRIDE = 16

demo_imgs = [to_float01(t.images[:t.length]) for t in demos]
chunk_centers, chunk_scaler = cluster_reference_chunks(
    demo_imgs,
    encoder,
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    use_standard_scaler=True,
)
print(f"Chunked cluster centers: {chunk_centers.shape}")
print(f"Window size: {WINDOW_SIZE}, stride: {STRIDE}")

In [ ]:
N_SAMPLE_CHUNK = 10

chunked_curves = {}
for label, trajs in [
    ("Demo", demos[:N_SAMPLE_CHUNK]),
    ("SFT Success", sft_success[:N_SAMPLE_CHUNK]),
    ("SFT Failed", sft_failed[:N_SAMPLE_CHUNK]),
    ("Random", random_trajs[:N_SAMPLE_CHUNK]),
]:
    curves = []
    for t in trajs:
        ts, dists = compute_chunked_progress_curve(
            to_float01(t.images[:t.length]),
            encoder, chunk_centers, chunk_scaler,
            window_size=WINDOW_SIZE, stride=STRIDE,
        )
        curves.append((ts, dists))
    chunked_curves[label] = curves
    mean_dist = np.mean([d.mean() for _, d in curves])
    print(f"{label:>12s}: {len(curves)} trajectories, mean chunk distance = {mean_dist:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
plot_chunked_progress_curves(
    chunked_curves,
    title=f"Exp 5b — Per-Chunk Distance (window={WINDOW_SIZE}, stride={STRIDE})",
    ax=ax,
)
plt.tight_layout()
plt.show()

In [ ]:
demo_imgs_sample = [to_float01(t.images[:t.length]) for t in demos[:20]]
chunk_corr, chunk_pval = compute_chunked_intra_trajectory_correlation(
    demo_imgs_sample, encoder, chunk_centers, chunk_scaler,
    window_size=WINDOW_SIZE, stride=STRIDE,
)
print(f"Intra-trajectory Spearman (demos, chunked): r={chunk_corr:.4f}, p={chunk_pval:.4f}")
print()
if chunk_corr < -0.5:
    print("Strong negative correlation: later chunks are closer to success clusters.")
    print("Chunked encoding captures progress within individual trajectories.")
elif chunk_corr < -0.2:
    print("Weak negative correlation: some progress signal, but noisy.")
else:
    print("No meaningful correlation: chunked encoding does not capture progress.")

---

## Experiment 6 — Embedding Dimensionality Analysis

**Hypothesis:** If 95% of variance is concentrated in very few dimensions,
the remaining dimensions are noise that inflates Euclidean distances and
makes all trajectories look equidistant. This would explain why
StandardScaler is critical — it normalises per-dimension variance so that
informative low-variance dimensions aren't drowned out.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

plot_pca_explained_variance(X_all, title="Exp 6a — PCA Explained Variance (all trajectories)", ax=axes[0])
plot_dimension_variance(X_all, title="Exp 6b — Per-Dimension Std Dev (top 50)", ax=axes[1])

plt.tight_layout()
plt.show()

cum_var, _ = pca_explained_variance(X_all)
for threshold in [0.90, 0.95, 0.99]:
    idx = np.searchsorted(cum_var, threshold) + 1
    print(f"{threshold*100:.0f}% variance captured by {idx} of {X_all.shape[1]} dimensions")

In [ ]:
dim_stats = embedding_dimension_stats(X_all)
print("Top 10 dimensions by std (highest variance):")
display(dim_stats.nlargest(10, "Std").style.format(precision=4))

print(f"\nDimension std range: {dim_stats['Std'].min():.6f} — {dim_stats['Std'].max():.6f}")
print(f"Max/Min std ratio: {dim_stats['Std'].max() / (dim_stats['Std'].min() + 1e-10):.1f}x")

---

## Experiment 7 — k-NN Elbow Plot

**Hypothesis:** A clear "elbow" (sharp knee) in the sorted k-NN distance
curve indicates natural cluster structure that DBSCAN can exploit. If the
curve is smooth with no elbow, the embedding space has no natural density
gaps and any eps choice is arbitrary.

We also compare k-NN curves between groups: if demo embeddings have a
lower, tighter curve than random embeddings, there's structure to exploit.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

plot_knn_elbow(
    X_reference, k_values=[2, 3, 5, 10],
    title="Exp 7a — k-NN Distances (reference embeddings, raw space)",
    ax=axes[0],
)

from sklearn.preprocessing import StandardScaler as SS
X_ref_scaled = SS().fit_transform(X_reference)
plot_knn_elbow(
    X_ref_scaled, k_values=[2, 3, 5, 10],
    title="Exp 7b — k-NN Distances (reference embeddings, scaled space)",
    ax=axes[1],
)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

knn_groups = {
    "Demo": X_demo,
    "SFT Success": X_sft_ok,
    "SFT Failed": X_sft_fail,
    "Random": X_random,
    "All References": X_reference,
}

plot_knn_elbow_comparison(knn_groups, k=2, title="Exp 7c — k-NN Curves by Group (k=2)", ax=ax)
plt.tight_layout()
plt.show()

---

## Experiment 8 — Progress Correlation Comparison

**Benchmark:** Top-Rewarder (Ding et al.) reports ~0.95 Spearman
correlation between VLM P("yes") scores and ground-truth task progress.
We compare our SRPO methods against this published reference.

**Methods compared:**
1. SRPO full-trajectory distance (from Exp 4)
2. SRPO chunked per-window distance (from Exp 5b)
3. Per-frame distance evolution (visualised below)
4. Top-Rewarder (published reference: r ~ 0.95)

### 8a — Per-Frame Distance Evolution

In [ ]:
N_SAMPLE = 10

per_frame_dists = {}
for label, trajs, centers, sc in [
    ("Demo", demos[:N_SAMPLE], centers_siirl, scaler),
    ("SFT Success", sft_success[:N_SAMPLE], centers_siirl, scaler),
    ("SFT Failed", sft_failed[:N_SAMPLE], centers_siirl, scaler),
    ("Random", random_trajs[:N_SAMPLE], centers_siirl, scaler),
]:
    dists_list = []
    for t in trajs:
        d = compute_per_frame_distances(
            t.images[:t.length], encoder, centers, sc, subsample_every=1,
        )
        dists_list.append(d)
    per_frame_dists[label] = dists_list

fig, ax = plt.subplots(figsize=(14, 6))
plot_per_frame_evolution(
    per_frame_dists,
    title="Exp 8a — Per-Frame Distance to Success Cluster Over Time",
    ax=ax,
)
plt.tight_layout()
plt.show()

### 8b — Multi-Method Correlation Table

Compares Spearman correlation (progress level vs reward signal) across
all methods tested in this notebook, with Top-Rewarder's published
~0.95 as a reference baseline.

In [ ]:
corr_full_traj, pval_full_traj = compute_progress_correlation(PROGRESS_LEVELS, distances_per_level_siirl)

methods = {
    "SRPO full-trajectory (siiRL)": (corr_full_traj, pval_full_traj),
    "SRPO chunked (window=32)": (chunk_corr, chunk_pval),
}

correlation_df = build_progress_correlation_table(methods, top_rewarder_ref=0.95)
print("Exp 8b — Progress Correlation Comparison:")
display(correlation_df.style.format({"Spearman r": "{:.4f}", "p-value": "{:.4f}"}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plot_progress_curve(
    PROGRESS_LEVELS,
    distances_per_level_siirl,
    title="SRPO Full-Trajectory Distance vs Progress",
    ax=axes[0],
)

plot_chunked_progress_curves(
    {k: v for k, v in chunked_curves.items() if k == "Demo"},
    title="SRPO Chunked (Demo) Distance Over Time",
    ax=axes[1],
)

plt.suptitle("Exp 8b — Full-Trajectory vs Chunked Progress Signal", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
print("Key takeaway:")
print(f"  Full-trajectory Spearman r = {corr_full_traj:.4f}")
print(f"  Chunked (w=32) Spearman r  = {chunk_corr:.4f}")
print(f"  Top-Rewarder (reference)    ~ 0.95")
print()
gap = 0.95 - max(abs(corr_full_traj), abs(chunk_corr))
if gap < 0.15:
    print("SRPO reward is competitive with Top-Rewarder for progress estimation.")
elif gap < 0.5:
    print("SRPO captures some progress signal but lags behind Top-Rewarder.")
else:
    print("SRPO progress signal is weak. The embedding may not capture task-relevant features.")

---

## Summary

Key questions to answer from the results above:

1. **Is there distance/reward separation between trajectory groups?**
   Check the KDE plots from Exp 1. If all groups overlap, the embedding
   is not capturing task-relevant information.

2. **Does StandardScaler help?**
   Compare the comparison table from Exp 2. Look for increased separation
   between success and failure groups.

3. **Which reward formula is better?**
   Compare reward ranges and standard deviations in Exp 3. The formula with
   wider dynamic range is preferable for RL.

4. **Is the progress reward monotonic?**
   Check the Spearman correlation from Exp 4. A strong negative correlation
   (r < -0.8) with low p-value confirms the reward captures task progress.

5. **Does clip encoding help?**
   Compare distance KDEs from Exp 5. If 64-frame clips show better
   separation than per-frame pooling, temporal context matters.

5b. **Does chunked encoding capture progress?**
   Check the intra-trajectory correlation in Exp 5b. Strong negative
   Spearman (r < -0.5) within individual demos means the chunked
   approach works as a dense progress reward.

6. **How concentrated is the variance?**
   Check Exp 6. If 95% variance is in <10 dims out of 1536, StandardScaler
   is essential and the effective embedding dimensionality is low.

7. **Is there natural cluster structure?**
   Check Exp 7. A clear elbow in the k-NN curve means DBSCAN can find
   meaningful clusters. No elbow = no natural density gaps.

8. **How does SRPO compare to Top-Rewarder?**
   Check the correlation table in Exp 8. Top-Rewarder achieves ~0.95
   Spearman correlation. If SRPO is near zero for both full-trajectory
   and chunked, the embedding is the fundamental bottleneck.